# Does TorchLens change fused attention?

`F.scaled_dot_product_attention` (SDPA) is one call, and its flash and memory-efficient
kernels never materialize the attention matrix. Two questions this notebook checks by
running the code rather than by reading about it:

1. Does `tl.trace` swap SDPA for an unfused path (or any other backend) to expose attention
   weights?
2. Does the traced forward's output differ numerically from the plain forward?

Method: for each SDPA backend the device offers (flash, memory-efficient, math), force that
backend for both a plain forward and a `tl.trace` forward, record which ATen kernel actually
ran (via the profiler), and compare the two outputs with `torch.equal` and the max absolute
difference. Runs on CPU and GPU.

In [ ]:
import importlib.util
import subprocess
import sys

# Skipped when both packages are already installed (e.g. a local checkout).
if (
    importlib.util.find_spec("torchlens") is None
    or importlib.util.find_spec("transformers") is None
):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "torchlens", "transformers"]
    )

In [ ]:
import os

os.environ.setdefault("KINETO_LOG_LEVEL", "5")  # quiet the profiler's own log lines

import torch
import torch.nn.functional as F
from torch.nn.attention import SDPBackend, sdpa_kernel

import torchlens as tl

device = "cuda" if torch.cuda.is_available() else "cpu"
print(
    "torch",
    torch.__version__,
    "| torchlens",
    tl.__version__,
    "| device",
    device,
    torch.cuda.get_device_name(0) if device == "cuda" else "",
)

BACKENDS = [SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH]
ACTIVITIES = [torch.profiler.ProfilerActivity.CPU] + (
    [torch.profiler.ProfilerActivity.CUDA] if device == "cuda" else []
)


def sdpa_kernels_seen(fn):
    """Run fn() under the profiler; return (fn's result, the SDPA ATen kernels that ran)."""
    with torch.profiler.profile(activities=ACTIVITIES) as prof:
        result = fn()
    names = sorted(
        {
            e.name
            for e in prof.events()
            if "scaled_dot_product" in e.name and e.name != "aten::scaled_dot_product_attention"
        }
    )
    return result, names


def compare(plain_fn, traced_fn, pick=lambda out: out):
    """Per SDPA backend: plain forward vs tl.trace output, the same backend forced for both."""
    last_trace, plain_by_backend = None, {}
    for backend in BACKENDS:
        try:
            with sdpa_kernel([backend]):
                plain, k_plain = sdpa_kernels_seen(plain_fn)
                trace, k_traced = sdpa_kernels_seen(traced_fn)
        except RuntimeError as err:
            print(f"{backend.name:19s} refused on this device: {str(err).splitlines()[0][:80]}")
            continue
        a, b = pick(plain), pick(trace.output)
        print(
            f"{backend.name:19s} torch.equal={torch.equal(a, b)}  max|plain-traced|={(a - b).abs().max().item():.3g}"
        )
        print(f"{'':19s} kernel plain={k_plain}")
        print(f"{'':19s} kernel traced={k_traced}")
        last_trace, plain_by_backend[backend.name] = trace, a
    names = list(plain_by_backend)
    for first, second in zip(names, names[1:]):
        diff = (plain_by_backend[first] - plain_by_backend[second]).abs().max().item()
        print(
            f"for scale: plain {first} vs plain {second} (PyTorch's own kernel difference) max|diff|={diff:.3g}"
        )
    return last_trace

## 1. A toy module with one fused SDPA call

Plain forward and traced forward, backend by backend. A backend that refuses on this device
(for example the memory-efficient kernel on CPU, or flash on an older GPU) is reported as such.

In [ ]:
class ToyAttention(torch.nn.Module):
    def __init__(self, d_model=64, n_heads=4):
        super().__init__()
        self.n_heads = n_heads
        self.qkv = torch.nn.Linear(d_model, 3 * d_model)
        self.proj = torch.nn.Linear(d_model, d_model)

    def forward(self, x):
        b, t, d = x.shape
        q, k, v = (
            self.qkv(x)
            .view(b, t, 3, self.n_heads, d // self.n_heads)
            .permute(2, 0, 3, 1, 4)
            .unbind(0)
        )
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # the one fused call
        return self.proj(y.transpose(1, 2).reshape(b, t, d))


torch.manual_seed(0)
toy = ToyAttention().to(device).eval()
x = torch.randn(
    2, 10, 64, device=device
)  # 10 tokens, head dim 16: an attention matrix would be (10, 10)

with torch.no_grad():
    trace = compare(lambda: toy(x), lambda: tl.trace(toy, x))

sdpa_ops = [op for op in trace.ops if "scaled_dot_product_attention" in str(op.func_name)]
op = sdpa_ops[0]
T = x.shape[1]
print()
print(
    f"{len(sdpa_ops)} SDPA op in the trace: {op.label}  saved output {tuple(op.out.shape)} {op.out.dtype}"
)
print("saved call arguments (default capture keeps none):", op.call.saved_args)
print("ops in the trace:", [o.label for o in trace.ops])
print(
    f"any saved tensor shaped like a ({T}, {T}) attention matrix?",
    any(isinstance(o.out, torch.Tensor) and tuple(o.out.shape[-2:]) == (T, T) for o in trace.ops),
)

## 2. GPT-2 with `attn_implementation="sdpa"`

Same comparison on the logits of a real model. Random weights are used only when the
pretrained download is unavailable; the comparison is the same either way.

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

try:
    gpt2 = GPT2LMHeadModel.from_pretrained("gpt2", attn_implementation="sdpa")
except Exception as err:
    print("pretrained weights unavailable, using random weights:", type(err).__name__)
    cfg = GPT2Config(n_layer=4)
    cfg._attn_implementation = "sdpa"
    gpt2 = GPT2LMHeadModel(cfg)
gpt2 = gpt2.to(device).eval()
torch.manual_seed(0)
ids = torch.randint(0, gpt2.config.vocab_size, (1, 24), device=device)
print("attn_implementation:", gpt2.config._attn_implementation, "| layers:", gpt2.config.n_layer)


def logits(out):
    return out.logits


with torch.no_grad():
    gtrace = compare(lambda: gpt2(ids), lambda: tl.trace(gpt2, ids), pick=logits)

attn_ops = [gtrace.ops[label] for label in gtrace.modules["transformer.h.0.attn"].calls[0].ops]
sdpa = [o for o in attn_ops if "scaled_dot_product_attention" in str(o.func_name)]
print()
print(
    f"transformer.h.0.attn holds {len(attn_ops)} ops, {len(sdpa)} of them SDPA:",
    sdpa[0].label,
    tuple(sdpa[0].out.shape),
)
print("ops of that module:", [o.func_name for o in attn_ops])

## 3. Attention weights, opt-in, from the saved tensors

TorchLens does not run an unfused attention to get the weights. After the capture, the
module's `facets` view reconstructs `scores`, `pattern` (the softmax matrix) and `z` on the
side from the Q, K and V the fused kernel actually received, and refuses to serve them unless
the recomputed `z` matches the kernel's saved output. They are read-only. Below, a plain
default-backend capture serves the pattern, and its forward output is compared with a plain
forward under the same default backend; nothing is re-run to get the weights.

In [ ]:
import copy

with torch.no_grad():
    ptrace = tl.trace(gpt2, ids)  # default capture, default SDPA backend, nothing forced
    again = gpt2(ids).logits
view = ptrace.modules["transformer.h.0.attn"].facets
print("declared facets:", {name: spec.status for name, spec in view.declared().items()})
pattern = view["pattern"].value  # also: gtrace.modules["transformer.h.0.attn"].facets.pattern
print(
    "pattern",
    tuple(pattern.shape),
    pattern.dtype,
    "| rows sum to 1:",
    torch.allclose(pattern.sum(-1), torch.ones_like(pattern.sum(-1)), atol=1e-5),
)
flags = view["pattern"].spec.capability_flags
print(
    "capability flags: read",
    flags.read,
    "| write",
    flags.write,
    "| reconstructed",
    flags.reconstructed,
)
print(
    "forward output of the capture that served the pattern == plain forward:",
    torch.equal(ptrace.output.logits, again),
)

# Cross-check against the eager attention weights Hugging Face returns for the same weights.
eager = copy.deepcopy(gpt2)
try:
    eager.set_attn_implementation("eager")
except AttributeError:
    eager.config._attn_implementation = "eager"
with torch.no_grad():
    hf_weights = eager(ids, output_attentions=True).attentions[0]
print(
    "max |reconstructed pattern - HF eager attention weights| =",
    (pattern - hf_weights).abs().max().item(),
)

## 4. The kernel underneath, and replay validation

`CaptureOptions(aten_ops=True)` records the ATen dispatches each op owns (off by default; the
default capture installs no dispatch mode). SDPA stays one op in the trace; the rows below are
the kernel it dispatched to. `tl.validate` replays every recorded op from its saved inputs,
SDPA included as one call, and compares against the saved outputs.

In [ ]:
flags_before = (
    torch.backends.cuda.flash_sdp_enabled(),
    torch.backends.cuda.mem_efficient_sdp_enabled(),
    torch.backends.cuda.math_sdp_enabled(),
)
with torch.no_grad():
    atrace = tl.trace(toy, x, capture=tl.options.CaptureOptions(aten_ops=True))
flags_after = (
    torch.backends.cuda.flash_sdp_enabled(),
    torch.backends.cuda.mem_efficient_sdp_enabled(),
    torch.backends.cuda.math_sdp_enabled(),
)
print(
    "global SDPA backend switches (flash, mem_efficient, math) before/after tracing:",
    flags_before,
    flags_after,
)

aop = next(o for o in atrace.ops if "scaled_dot_product_attention" in str(o.func_name))
print(f"ATen dispatches owned by the one SDPA op ({aop.label}):")
for row in aop.aten_ops:
    print("   ", f"{row.namespace}::{row.operator}.{row.overload}")

print("tl.validate(toy, x) ->", tl.validate(toy, x))

## What to look for

- Every backend row prints `torch.equal=True` and `max|plain-traced|=0`: the traced forward
  is the plain forward, bit for bit, and the same kernel ran in both.
- The "for scale" line shows that different PyTorch kernels do differ from each other by float
  rounding. That is PyTorch's backend choice; tracing does not change which one runs.
- The trace holds one SDPA op with its output; no `(T, T)` attention matrix is saved.
- The `pattern` facet is reconstructed afterwards from the kernel's own inputs, matches
  Hugging Face's eager attention weights, is read-only, and leaves the forward untouched.
- On a GPU that refuses flash the memory-efficient kernel runs instead; the kernel names printed
  per row say which one did. PyTorch's flash kernel needs compute capability 8.0 or later, so an
  RTX 2080 Ti (7.5) or a T4 refuses it; measured on a 2080 Ti, the memory-efficient and math rows
  match bit for bit. There PyTorch also refuses the forced memory-efficient kernel for the toy
  module's plain forward under `torch.no_grad()`, before any tracing; the GPT-2 row runs it.